# TabFM pentru semnalul A, pe GPU

Ruleaza cele 12 potriviri ale bratului A cu TabFM (Google Research) si scrie predictiile
inapoi, ca sa fie consumate local cu `--regressor tabfm-cache`.

## Ce trebuie sa fie in Drive inainte de a rula

Intr-un folder al tau, implicit `MyDrive/tabfm/`:

```
tabfm_bridge.py                 din radacina proiectului, versiunea CURENTA
tabfm_blocks_proj394.npz        produs local cu --export-blocks
```

Comanda locala care produce blocurile:

```
python finetune_xlmr.py --frozentrial --lemma-freq --afeat fert_inloc \
    --emb-tf proj-394 --export-blocks data/tabfm_blocks_proj394.npz
```

Celula 4 verifica functional ca `tabfm_bridge.py` din Drive e cel curent, nu o copie veche.
Copia veche e o capcana reala: nu crapa, doar ii lipsesc butoanele si porti, si rularea
merge inainte fara ele.

## De ce nu se ruleaza pe latimea intreaga

X-ul bratului A are 774 de coloane, iar TabFM e documentat pentru maximum 500. In plus, costul
lui masurat creste SUPRA-PATRATIC in numarul de coloane (masurat pe M5: la 2000 de randuri, 100
coloane 25s, 200 coloane 68s, 400 coloane 441s) si doar LINIAR in randuri (2000 -> 4000 la 400
de coloane: 441s -> 850s). Deci latimea e parghia de cost, nu adancimea setului.

`proj-394` pastreaza 95.1% din varianta embeddingului si costa incumbentul `blend` doar 0.0029.
Comparatia se face numai fata de controlul rulat pe ACELASI spatiu, niciodata fata de cifra pe
spatiul plin.

## Rezultatul

Un `.npy` per cerere, numit cu cheia de identitate a cererii. Cheia acopera formele si octetii
lui `(Xtr, ytr, Xpredict)`, deci daca notebook-ul ar reconstrui fie si un rand altfel decat
codul local, fisierul n-ar corespunde si importul local se opreste.

## 1. Ce GPU am primit, si de ce conteaza mai mult decat pare

Doua cifre decid totul aici, si niciuna nu e viteza.

**Modelul are 1.65 miliarde de parametri.** Configuratia reala din checkpoint nu seamana cu
implicitele din semnatura lui `TabFM.__init__`: `embed_dim=256` (nu 8), `icl_num_blocks=24`
(nu 2), `icl_dim=2048`, FFN 8192. `model.safetensors` are **6.59 GB**, adica 6.14 GiB de
greutati pe placa in float32 si 3.07 GiB in bfloat16.

**Fiecare etapa tine un tensor `[1, T, HC, 256]`**, cu `T = 14107` randuri (context plus de
prezis) si `HC` coloane. Notand `A` marimea lui, `ColEmbedding.forward` are patru astfel de
tensori vii deodata: intrarea, copia facuta de `permute().reshape()`, lista de bucati a lui
`torch.cat` si iesirea.

De aici iese verdictul pe un T4 de 14.6 GiB utilizabili:

```
float32 (greutati 6.14 GiB):
  HC=320 -> A=4.30 GiB, 4A+greutati = 23.4 GiB   OOM sigur
  HC=128 -> A=1.72 GiB, 4A+greutati = 13.0 GiB   la limita

bfloat16 (greutati 3.07 GiB):
  HC=320 -> A=2.15 GiB, 4A+greutati = 11.7 GiB   plauzibil
  HC=256 -> A=1.72 GiB, 4A+greutati = 10.0 GiB   confortabil
```

**Deci pe T4 float32 nu e o alternativa mai lenta dar sigura: nu incape la nicio latime
utila.** bfloat16 e obligatoriu, si patch-urile din celula 6 duc 4A la 2A.

### Si atunci float16?

Nu. In float16 TabFM da NaN pe orice placa. Masurat pe date sintetice, acelasi checkpoint:
float32 si bfloat16 dau 150 din 150 de predictii finite si difera intre ele cu 0.016, float16
da **0 din 150**. Localizat cu hook-uri pe module: reziduul stivei ICL ajunge la ~4300 pana la
`icl_predictor.tf_icl.blocks.9.attn.out_proj`, iar `linear2` din FFN trece de 65504, maximul
lui fp16. Prima valoare `inf` apare fix acolo. Asta era eroarea

```
SystemExit: [tabfm] ca: 505 predictii pentru 505 randuri, finite 0
```

`tabfm_bridge.py` refuza fp16 chiar daca i-l ceri.

### Ce ramane incert pana la prima rulare

bfloat16 pe Turing (T4, capability 7.5) nu are unitati dedicate, si cuBLAS cere capability 8.0
pentru GEMM bf16. Poate merge prin emulare, poate crapa. Podul afla in cateva secunde, pe date
sintetice, inainte de orice cost. Daca nu merge, nicio latime nu salveaza rularea pe T4, si
raspunsul e un runtime L4 sau A100 (capability 8.9 / 8.0).

In [2]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

Wed Aug 26 11:06:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Drive: se monteaza ACUM, nu dupa instalare

`HF_HOME` trebuie sa fie pe Drive **inainte de primul `import tabfm`** de mai jos, nu dupa cum
statea inainte. Motivul, verificat pe sursa reala din `tabfm` si `huggingface_hub`, nu presupus:
`tabfm/__init__.py` importa la nivel de modul `tabfm.src.pytorch.tabfm_v1_0_0`, care la randul lui
face `from huggingface_hub import ... constants ...`. `huggingface_hub/constants.py` calculeaza
`HF_HOME` (si `HF_HUB_CACHE`, folosit implicit la descarcarea checkpointului) ca pe o **constanta
de modul**, o singura data, din variabila de mediu de la momentul importului. Daca `HF_HOME` s-ar
seta dupa `import tabfm`, valoarea deja inghetata ar ramane cea implicita (`~/.cache/huggingface`,
efemer in Colab), iar caching-ul in Drive al celor 6.59 GB n-ar functiona NICIODATA: fiecare
sesiune noua ar redescarca tot checkpointul, tacut, fara nicio eroare care sa arate de ce.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
LUCRU = '/content/drive/MyDrive/tabfm'      # schimba daca ai pus fisierele altundeva

# Trebuie sa fie setat AICI, inainte de `pip install`/`import tabfm` din celula urmatoare: vezi
# nota de mai sus. `os.makedirs` e obligatoriu inainte ca huggingface_hub sa scrie ceva sub el.
os.environ['HF_HOME'] = os.path.join(LUCRU, 'hf')
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/tabfm/hf


## 3. Instalare

`safetensors` se cere explicit: `huggingface_hub` il foloseste la incarcarea checkpointului dar
nu il trage ca dependenta a lui `tabfm`, iar fara el `load()` crapa cu `NameError`.

In [4]:
import os
# Se pune INAINTE de `import torch`: cu alocari de ordinul gigaoctetului, segmentele
# extensibile sunt diferenta dintre a incapea si a da OOM din fragmentare.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Versiune PINUITA. Patch-urile de memorie din celula 6 sunt copiate din 1.0.1 si verificate
# bit-identic fata de ea. La o versiune noua poarta lor cade si se ruleaza originalul, ceea ce
# inseamna varf de memorie cu ~50% mai mare, deci posibil OOM. Ridici pinul deliberat, nu din
# intamplare.
!pip install -q "tabfm[pytorch]==1.0.1" safetensors

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
import tabfm
print('tabfm', getattr(tabfm, '__version__', 'necunoscut'))
if not torch.cuda.is_available():
    raise SystemExit('CUDA nu se vede. Ori runtime-ul nu are GPU (Runtime > Change runtime type), '
                     'ori instalarea a stricat torch si trebuie restartat runtime-ul.')

# `tabfm` cere jaxtyping<0.3 si typeguard<3, iar Colab vine des cu versiuni mai noi. Daca pip a
# scris "You must restart the runtime", restarteaza si RE-RULEAZA celula asta inainte de a merge
# mai departe: altfel importurile de mai jos iau pachetele vechi din memorie.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.
torch 2.11.0+cu128 | cuda True
tabfm 1.0.1


## 4. Drive: verificarea podului

Drive e deja montat (celula 2) si `HF_HOME` deja setat. Celula asta doar defineste caile ramase
si verifica pe FUNCTII ca `tabfm_bridge.py` din Drive e cel curent, nu o copie veche: rularea
scrie dupa FIECARE cerere si sare peste ce exista deja, deci daca sesiunea cade, reluarea
continua de unde a ramas.

Verificarea nu e pe suma de control, ci pe FUNCTII: o copie veche a lui `tabfm_bridge.py` nu
crapa, doar ii lipsesc `TABFM_CHUNK` si poarta de dtype, si atunci esti inapoi la eroarea de sus
fara sa stii de ce. Daca pica, recopiaza fisierul din proiect.

In [5]:
import sys, importlib, inspect

BLOCURI = os.path.join(LUCRU, 'tabfm_blocks_proj394.npz')
# Sufix `_maxfeat256`: la MAXFEAT=160 s-a scris deja `preds_proj394` (rulare terminata, 12/12).
# `ruleaza()` refuza sa amestece doua configuratii in acelasi DEST (poarta pe `setari.json`),
# deci orice alt buton schimbat cere alt DEST. Cele doua seturi raman comparabile intre ele.
DEST = os.path.join(LUCRU, 'preds_proj394_maxfeat256')

# Checkpointul are 6.59 GB si se descarca de la HuggingFace la FIECARE sesiune noua daca
# HF_HOME (setat in celula 2, INAINTE de import tabfm) nu are deja o copie cache-uita.
#
# NU e garantat un castig: Drive montat citeste fisiere mari cu ~10-40 MB/s, iar descarcarea
# de la HuggingFace merge des la 50-200 MB/s. Celula 7 tipareste timpul de incarcare; daca
# iese mai prost decat descarcarea, comenteaza randul cu `HF_HOME` din celula 2 si compara.

sys.path.insert(0, LUCRU)                   # ca sa gaseasca tabfm_bridge.py
for p in (os.path.join(LUCRU, 'tabfm_bridge.py'), BLOCURI):
    if not os.path.exists(p):
        raise SystemExit(f'lipseste {p}')
    print(f'{p}  {os.path.getsize(p)/1e6:.1f} MB')

# reload, nu doar import: daca ai recopiat fisierul in Drive dupa ce ai rulat celula o data,
# fara reload ai in memorie tot versiunea veche.
import tabfm_bridge
importlib.reload(tabfm_bridge)
lipsa = [n for n in ('proba_finita', 'parametri_ansamblu', 'cereri', 'req_key', 'ruleaza',
                    'instaleaza_patch_uri', 'checkpoint_membri')
         if not hasattr(tabfm_bridge, n)]
if 'TABFM_CHUNK' not in inspect.getsource(tabfm_bridge.ruleaza):
    lipsa.append('TABFM_CHUNK')
# Garda care lipsea, si care a costat doua rulari. Patch-ul de atentie e singurul lucru care
# tine T=14107 pe o placa de 15 GiB, iar o copie veche a fisierului nu crapa: doar il pierde,
# si afli abia din OOM-ul primei cereri, dupa ce ai platit incarcarea modelului.
if (hasattr(tabfm_bridge, 'instaleaza_patch_uri')
        and '_attn_econom' not in inspect.getsource(tabfm_bridge.instaleaza_patch_uri)):
    lipsa.append('_attn_econom')
# Aceeasi garda, pentru checkpointul pe MEMBRU de ansamblu. O copie veche nu crapa: rularea merge
# mai departe protejata doar intre cereri, iar la ~42 min pe cerere o taiere de sesiune arde tot.
if not hasattr(tabfm_bridge, 'proba_checkpoint'):
    lipsa.append('proba_checkpoint')
elif (hasattr(tabfm_bridge, 'instaleaza_patch_uri')
        and '_bf_ckpt' not in inspect.getsource(tabfm_bridge.instaleaza_patch_uri)):
    lipsa.append('_bf_ckpt')
if lipsa:
    raise SystemExit(f'tabfm_bridge.py din Drive e o versiune veche, ii lipseste {lipsa}. '
                     f'Recopiaza-l din radacina proiectului.')
print('tabfm_bridge: versiunea curenta')

/content/drive/MyDrive/tabfm/tabfm_bridge.py  0.0 MB
/content/drive/MyDrive/tabfm/tabfm_blocks_proj394.npz  23.9 MB
tabfm_bridge: versiunea curenta


## 5. Parametrii rularii

**`n_estimators` costa TIMP, nu memorie.** Asta contrazice sfatul care era scris aici inainte.
`TabFMRegressor` are `batch_size=1` (`classifier_and_regressor.py`, `__init__` si
`_batch_forward`), deci membrii ansamblului trec prin model unul cate unul: varful de memorie e
al unui SINGUR membru, oricat de mare ai pune `n_estimators`. Daca da CUDA out of memory,
`n_estimators` nu e butonul; butoanele sunt `TABFM_CHUNK`, `TABFM_MAXFEAT` si numarul de randuri
de context.

Ce face `n_estimators`: fiecare membru vede acelasi context, dar cu alta permutare de coloane,
alta normalizare si alt subesantion, iar predictiile se mediaza. Castigul e reducere de varianta
si se aplatizeaza; implicitul lor e 32.

### Cele trei feluri de butoane

**Gratis.** `TABFM_NORM` imparte membrii peste mai multe familii de preprocesare in loc de doua.
Nu adauga nicio trecere prin model, doar face membrii mai putin corelati intre ei, ceea ce e
exact ce mediaza mai bine. Optiuni: `none`, `power`, `quantile`, `quantile_rtdl`, `robust`.

**Mai bune SI mai ieftine.** `TABFM_MAXFEAT` sub latimea reala face fiecare membru sa vada alt
subesantion de coloane. Costul in coloane e supra-patratic (exponent 2.70, masurat de tine intre
200 si 400), deci 320 din 400 costa 0.56 din timp si in acelasi timp decoreleaza membrii. La
buget fix incap aproape de doua ori mai multi. Riscul: fiecare membru e mai slab, deci nu cobori
oricat.

**Scumpe.** `TABFM_CROSSES` si `TABFM_SVD` adauga coloane, dar numai la membrii impari
(alocarea "split" din `_get_member_n_features_list`), deci ~jumatate din ce pare. `TABFM_NNLS`
pondereaza membrii in loc sa-i medieze si costa aproape inca un pasaj complet, adica ~2x.

### Ce nu se atinge

`TABFM_MAX_ROWS` se lasa gol: rularea foloseste TOT contextul. Daca il pui, blocul de antrenare
nu mai e acelasi cu al celorlalte brate si subesantionarea trebuie declarata oriunde e citata
cifra. In plus, `tabfm` refuza explicit `max_num_rows` impreuna cu NNLS.

`feat_shuffle_method` are doar `random` si `none`, si e deja pe `random`. Nu e un buton.

### Cum nu amesteci doua rulari

Toate butoanele de mai sus intra in `setari.json` din `DEST` si sunt comparate la fiecare
reluare, deci nu poti amesteca din greseala doua configuratii in acelasi set de predictii. Ca sa
rulezi alta configuratie, dai alt `DEST`, si atunci ai doua seturi comparabile intre ele. Daca
vrei sa atribui un castig unui buton anume, schimba **un singur** buton intre doua `DEST`-uri.

In [6]:
cap = torch.cuda.get_device_capability()
gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'capability {cap}, {gib:.1f} GiB')

# Singura intrebare deschisa pe placa asta: bf16 pe Turing (7.5) n-are unitati dedicate si
# cuBLAS cere 8.0 pentru GEMM bf16. Raspunsul costa o secunda, deci se ia INAINTE de cele
# 6.59 GB de checkpoint, nu dupa. Daca pica aici, nicio latime nu salveaza rularea pe T4.
if cap[0] < 8:
    import time
    a = torch.randn(2048, 2048, device='cuda')
    try:
        r32 = a @ a
        rel = ((a.bfloat16() @ a.bfloat16()).float() - r32).abs().max() / r32.abs().max()
        def _cron(f):
            f(); torch.cuda.synchronize(); t = time.time()
            for _ in range(10):
                f()
            torch.cuda.synchronize(); return time.time() - t
        raport = _cron(lambda: a.bfloat16() @ a.bfloat16()) / _cron(lambda: a @ a)
        print(f'bf16 GEMM merge: eroare relativa {rel:.1e}, {raport:.1f}x timpul lui float32')
        print('  asteptat 1e-3..1e-2 si ~1x. O eroare de ~1e-7 inseamna ca torch a urcat tacit'
              '\n  calculul in float32: corect, dar fara castigul de viteza si de memorie.')
        if raport > 1.5:
            print(f'  bf16 e emulat si costa {raport:.1f}x: bugetul de timp de mai jos, care'
                  '\n  vine de pe M5, iese optimist cu acelasi factor.')
    except Exception as e:
        print(f'bf16 GEMM CADE pe capability {cap}: {type(e).__name__}: {e}')
        print('  Asta inchide T4-ul: float32 nu incape la nicio latime utila (greutatile'
              '\n  singure 6.14 GiB). Runtime L4 sau A100 (capability 8.9 / 8.0).')
    del a, r32
    torch.cuda.empty_cache()

# dtype: bfloat16 pe ORICE placa. Pe T4 float32 NU e alternativa, contrar a ce parea:
# checkpointul are 1.65 miliarde de parametri, deci 6.14 GiB doar greutatile in fp32, si
# fiecare tensor de etapa e T x HC x 256 x 4 octeti. La T=14107 nu incape la nicio latime
# utila. In bf16 greutatile scad la 3.07 GiB si tensorii la jumatate.
# Gol = implicitul podului (bfloat16), cu coborare automata daca proba nu trece.
os.environ['TABFM_DTYPE'] = ''

# Chunking pe activari. Implicitul lor, 4096, e ales "for memory safety on a 40 GB GPU".
os.environ['TABFM_CHUNK'] = '1024'

# Bugetul unei matrice de scoruri de atentie, in GiB. Peste el, patch-ul de atentie din
# `tabfm_bridge` taie pe intervale de query. Butonul asta, nu TABFM_MAXFEAT, e cel care raspunde
# la un OOM din stiva ICL: acolo latimea e deja colapsata si singurul lucru care conteaza e T.
os.environ['TABFM_ATTN_GIB'] = '0.5'

# --- bugetul: A DOUA rulare, MAXFEAT urcat de la 160 la 256 -----------------------------
# Reperul REAL, masurat pe ACEST T4, nu extrapolat de pe M5: rularea de la MAXFEAT=160 a facut
# 12 cereri in 142 min (n_estimators=8), adica 743s/cerere la T~14107. Formula de mai jos
# extrapoleaza pe COLOANE cu exponentul 2.70 masurat pe M5 (neverificat inca pe T4 pe axa
# coloanelor — celula 7 recalibreaza doar pe randuri, la MAXFEAT-ul curent):
#
#     s/cerere la MAXFEAT=256 =~ 743 x (256/160)^2.70 =~ 2640s (~44 min)
#
# La n_estimators=8: ~44 min/cerere, ~8.8 ore pentru toate 12 — PESTE pragul de 30 min/cerere,
# deci o deconectare in mijlocul unei cereri o pierde intreaga (nu si cererile deja scrise).
# Memoria: activarile la MAXFEAT=160 au fost 5.45 GiB (masurat, celula 7). Daca scaleaza liniar
# cu HC (asa cum promit patch-urile, tensor [1,T,HC,256]): ~5.45 x 256/160 =~ 8.7 GiB activari,
# deci varf ~11.8 GiB din 14.6 — sub 92%, dar NEVERIFICAT: ruleaza din nou celula 7 dupa
# schimbarea asta, INAINTE de celula 8, ca sa confirmi cifra reala, nu extrapolarea.
#
# Daca 8.8 ore nu incape intr-o cota saptamanala Colab (gratuit: ~15-30 ore/saptamana,
# dinamic), scade `n_estimators` la 4 (~4.4 ore) sau accepta 2-3 sesiuni separate: `ruleaza()`
# scrie dupa fiecare cerere si sare peste cele gata, deci reluarea intre sesiuni e sigura.
os.environ['TABFM_ESTIMATORS'] = '8'
os.environ['TABFM_MAX_ROWS'] = ''            # gol = tot contextul

# GRATIS pe GPU. `EnsembleGenerator` imparte cei n_estimators membri peste lista de
# normalizari, nu adauga treceri prin model. De la 2 familii de preprocesare la 4, aceiasi
# bani. Costa doar CPU la `fit`, unde `power` si `quantile_rtdl` sunt cele lente.
os.environ['TABFM_NORM'] = 'none,power,quantile_rtdl,robust'

# Urcat de la 160 la 256: fiecare membru vede mai multe coloane, deci pe A singur ar trebui
# sa urce usor (era 0.5541 la 160 pe proj-394). Riscul stabilit deja pe axa asta: rho fata de
# ridge/hgb NU a scazut de la proj-200 la proj-394 (a urcat, de fapt), deci nu exista o
# tendinta masurata care sa spuna ca 256 va decorela mai bine — doar ca va costa mai mult.
os.environ['TABFM_MAXFEAT'] = '256'

# Produse de perechi, doar la membrii impari (alocarea "split"), deci ~jumatate din cat pare.
os.environ['TABFM_CROSSES'] = 'sqrt'

# SVD peste coloanele originale. Pe proj-394 spatiul e DEJA o proiectie ortogonala, deci
# componentele ar re-deriva aceleasi directii. Ieftin de incercat separat.
os.environ['TABFM_SVD'] = '0'

# Aproape inca un pasaj complet peste toti membrii (~2x). Nu incape in bugetul de o sesiune.
os.environ['TABFM_NNLS'] = '0'

os.environ['TABFM_OUTLIER'] = '4.0'          # pragul de z la care se taie cozile

capability (7, 5), 14.6 GiB
bf16 GEMM merge: eroare relativa 3.8e-03, 1.8x timpul lui float32
  asteptat 1e-3..1e-2 si ~1x. O eroare de ~1e-7 inseamna ca torch a urcat tacit
  calculul in float32: corect, dar fara castigul de viteza si de memorie.
  bf16 e emulat si costa 1.8x: bugetul de timp de mai jos, care
  vine de pe M5, iese optimist cu acelasi factor.


## 6. Patch-urile de memorie

Patru patch-uri de memorie plus un al cincilea de supravietuire, si toate stau acum in
`tabfm_bridge.py`, nu aici. Motivul e o rulare pierduta:
cand stateau in celula asta, celula s-a desincronizat tacit de fisierul din proiect, rularea a
mers cu atentia nepatch-uita si a dat OOM, iar traceback-ul arata `model.py:129`, adica
biblioteca, nu celula. Acum `ruleaza` le instaleaza singur si refuza sa porneasca fara cel de
atentie, iar celula 4 verifica pe sursa ca fisierul din Drive chiar il contine.

Cel care decide daca incape e **atentia**. `ICLearning` ruleaza tot contextul ca o singura
secventa si atentia lui e peste RANDURI, cu 16 capete, deci matricea de scoruri e
`[1, 16, T, T]`: la T=14107 in bfloat16 sunt 5.93 GiB, iar drumul `math` al lui SDPA, ales
fiindca `attn_mask` e tensor dens, tine doua copii deodata. Termenul e PATRATIC in T si
`TABFM_MAXFEAT` nu-l atinge deloc, fiindca latimea e colapsata inainte de stiva ICL.

Celelalte trei taie copii ale tensorului de etapa `[1, T, HC, 256]`: `CellEmbedder` prin operatii
pe loc, `ColEmbedding` de la 4A la 2A, `RowInteraction` de la 3A la 2A.

Pragurile difera deliberat. Cele trei de memorie trebuie sa fie **exact zero** fata de original.
Atentia nu poate: taierea pe query e exacta algebric, dar SDPA alege alt nucleu pentru alta
forma, deci acumularea difera in ultimii biti. Masurat pe 64 de cazuri in float32: 7.2e-07, cu
patru ordine de marime sub precizia lui bfloat16. Pragul ei e 1e-5, si caderea lui opreste
rularea.

Al cincilea, `TabFMRegressor._batch_forward`, nu e de memorie: pune **checkpoint pe membru de
ansamblu**. `batch_size` e 1, deci membrii trec prin model unul cate unul si abia la final se
concateneaza, in interiorul unui singur apel `predict`. La MAXFEAT=160 o cerere dura 12 minute si
scrisul per cerere ajungea; la 256 trece de 40, peste pragul de siguranta de 30, si o sesiune
Colab taiata la mijloc ar arde tot. Acum fiecare membru se scrie pe disc cand se termina, iar la
reluare se citesc cei gata si se continua cu restul. Nu se poate verifica fata de original pe
tensori mici, fiindca intrebarea nu e daca aritmetica e aceeasi, ci daca membrul `i` regenerat
maine e ACELASI: poarta lui e `proba_checkpoint`, care ruleaza aceeasi predictie de patru ori
(fara checkpoint, cu director gol, dupa stergerea ultimului fisier, si cu toate pe disc) si cere
`array_equal` intre toate patru plus zero treceri prin model la reluarea completa.

Celula de mai jos e optionala PENTRU RULAREA REALA (celula `ruleaza` isi instaleaza singura
patch-urile), dar celula 7 de calibrare NU trece prin `ruleaza` si isi instaleaza si ea
patch-urile la inceput, tocmai ca sa nu depinda de faptul ca ai rulat-o pe asta inainte.
Ruleaz-o daca vrei doar sa vezi cifrele devreme, inainte sa platesti incarcarea modelului.

In [7]:
from tabfm_bridge import instaleaza_patch_uri

# `ruleaza` cheama asta singur, deci rularea NU depinde de faptul ca ai rulat celula. Aici e doar
# ca sa vezi devreme cele doua cifre. A doua chemare in acelasi proces re-verifica pe original,
# nu pe patch, deci poti re-rula fara sa golesti verificarea.
instaleaza_patch_uri()


[tabfm] patch-uri: atentie 9.5e-07 (prag 1e-5, buget 0.5 GiB), memorie exact 0, checkpoint pe membru instalat


{'atentie': 9.5367431640625e-07, 'memorie': 0.0, 'attn_gib': 0.5, 'ckpt': True}

## 7. Cat costa, inainte sa coste

Celula asta e verificarea propriu-zisa, si e singura care poate raspunde cu cifre in locul
aritmeticii de mai sus. Isi instaleaza singura patch-urile de memorie (nu depinde de celula 6),
incarca modelul, masoara cat cantaresc greutatile pe placa, apoi ruleaza aceeasi cerere de doua
ori, la doua adancimi de context, cu **exact configuratia setata mai sus** si `n_estimators=2`.

Doi, nu unu, si e o corectie: `TABFM_CROSSES=sqrt` da 16 coloane in plus doar membrilor impari,
iar lotul se umple apoi la latimea celui mai lat, deci la MAXFEAT=256 modelul vede 272. Cu un
singur membru nu exista niciun membru impar si calibrarea masura 256, adica 6.25% mai putin
decat rularea adevarata, fix pe termenul care decide daca incape.

Ce scoate:

- **timpul de incarcare a modelului.** E reperul pentru decizia cu `HF_HOME` in Drive: daca e
  mai prost decat descarcarea de la HuggingFace, comenteaza randul din celula 2.
- **varful de memorie, separat pe greutati si activari.** Doar activarile cresc cu `T`, deci
  numai ele se extrapoleaza; greutatile se adauga la loc dupa. Daca nu incape, iti spune la ce
  valoare sa cobori `TABFM_MAXFEAT`.
- **timpul per membru, per cerere si pe toate 12.** `fit` (preprocesare pe CPU) si `predict`
  (partea de GPU) se cronometreaza separat, fiindca doar al doilea se inmulteste cu
  `n_estimators`. Cifra de urmarit e cea per MEMBRU: reperul de la MAXFEAT=160 e ~1.5 min, si
  asteptarea la 256, cu exponentul 2.70 pe latimile reale 172 -> 272, e ~5.3 min.

Un prag care s-a mutat. **Per cerere** era ~30 de minute, fiindca o cerere intrerupta se pierdea
intreaga. Cu checkpointul pe membru pragul e acum **per membru**, si el se respecta usor. **Per sesiune:** ordinea de
rulare pune limbile primele si bratele EN-only ultimele, ca o sesiune care cade sa lase pe disc
partea de care are nevoie poarta de frontiera.

Reperul cu care compari ce iese: rularea locala incheiata a facut 12 cereri in 79 de minute la
206 coloane si `n_estimators=2`, adica **3.3 minute pe membru pe cerere**. Formula care a dat
configuratia din celula 5 e

```
min/cerere = 3.3 x (MAXFEAT/206)^2.70 x n_estimators
```

si prezice ~13 min pe cerere la 160 de coloane cu 8 membri, ~2.7 ore in total. Daca placa
primita iese mult peste, ai un T4 mai lent decat M5-ul si cobori `n_estimators`; daca iese mult
sub, urci.

Nu rula celula 8 inainte sa te uiti la ce iese aici.

In [9]:
import time, numpy as np, torch
from tabfm import TabFMRegressor
from tabfm.src.pytorch.tabfm_v1_0_0 import load
from tabfm_bridge import cereri, parametri_ansamblu, instaleaza_patch_uri

# Celula asta NU trece prin `ruleaza`, deci isi instaleaza singura patch-urile de memorie: fara
# asta, atentia nepatch-uita cere [1, 16, T, T] la T=14107 (~12 GiB pe drumul `math` al SDPA) si
# calibrarea insasi ar crapa cu OOM la al doilea punct, inainte sa apuce sa spuna "NU INCAPE".
# Idempotenta: sigur de rulat chiar daca ai rulat deja celula 6 in acelasi proces.
instaleaza_patch_uri()

B = np.load(BLOCURI)
reqs = cereri(B)
T = {nume: len(Xtr) + len(Xp) for nume, Xtr, _, Xp in reqs}
LAT = reqs[0][1].shape[1]
ans = parametri_ansamblu()
HC = ans['max_num_features'] or LAT
print(f'latime X: {LAT} coloane, din care un membru vede {HC}')
print('ansamblu:', ans)
print('cele 12 cereri, lungimea secventei (context + de prezis):')
for nume, t in sorted(T.items(), key=lambda kv: -kv[1]):
    print(f'  {nume:14s} {t:6d}')

TOTAL_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
dt = getattr(torch, os.environ['TABFM_DTYPE'] or 'bfloat16')
model = load(model_type='regression', device='cuda', dtype=dt)
t_load = time.time() - t0
BAZA = torch.cuda.memory_allocated() / 2**30      # greutatile: NU cresc cu T
print(f'\nincarcare model: {t_load:.0f}s  |  greutati pe placa {BAZA:.2f} GiB din {TOTAL_GIB:.1f}')
print('  (timpul asta e reperul pentru HF_HOME in Drive vs descarcare directa)')

r = int(os.environ['TABFM_CHUNK'])
for mod in model.modules():
    if hasattr(mod, 'row_chunk_size'):
        mod.row_chunk_size = r
    if hasattr(mod, 'ffn_chunk_size'):
        mod.ffn_chunk_size = max(2 * r, 512)

nume, Xtr, ytr, Xp = reqs[0]
med = np.nanmedian(Xtr, axis=0)
med = np.where(np.isfinite(med), med, 0.0)
A = np.where(np.isfinite(Xtr), Xtr, med).astype(np.float32)
P = np.where(np.isfinite(Xp), Xp, med).astype(np.float32)
b = np.asarray(ytr, np.float32)

N_CAL = 2      # <- NU 1, si asta e o corectie, nu o preferinta.
# `TABFM_CROSSES=sqrt` da coloane in plus DOAR membrilor impari (alocarea "split" din
# `_get_member_n_features_list`), iar `_transform_features` umple apoi TOTI membrii la latimea
# celui mai lat. Masurat direct pe `EnsembleGenerator` cu MAXFEAT=256:
#
#     n_estimators=1  ->  Xs_all (1, T, 256)      k_crosses = [0]
#     n_estimators=2  ->  Xs_all (2, T, 272)      k_crosses = [0, 16]
#     n_estimators=8  ->  Xs_all (8, T, 272)      k_crosses = [0, 16, 0, 16, 0, 16, 0, 16]
#
# Cu un singur membru nu exista niciun membru impar, deci calibrarea masura la 256 memoria unei
# rulari care merge de fapt la 272: subestimare de 6.25% fix pe partea care decide daca incape.
# Doi membri sunt minimul care aduce latimea adevarata in lot. Costa inca o trecere prin model.
masurat = []
rng = np.random.default_rng(0)
# Al doilea punct e cererea INTREAGA, nu unul mic din care se extrapoleaza. Versiunea de dinainte
# masura la T=1700 si 3200 si extrapola de 4.4x, si a ratat exact termenul care conteaza:
# atentia ICL tine [1, nhead, T, T], deci creste cu T^2 si la T mic nu se vede. A raportat
# "incape" pentru o configuratie care a dat OOM la prima cerere adevarata. Un punct la T real
# costa un minut si nu se poate insela.
HC_REAL, N_NORM_CAL = None, None
for n_ctx, npred in ((3000, 200), (len(A), len(P))):
    sel = rng.choice(len(A), n_ctx, replace=False) if n_ctx < len(A) else np.arange(len(A))
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    reg = TabFMRegressor(model=model, n_estimators=N_CAL,
                         max_num_features=ans['max_num_features'] or A.shape[1],
                         norm_methods=ans['norm_methods'],
                         n_feature_crosses=ans['n_feature_crosses'],
                         n_svd_features=ans['n_svd_features'],
                         enable_nnls=False,          # NNLS costa ~2x si se adauga separat mai jos
                         outlier_threshold=ans['outlier_threshold'],
                         random_state=42, verbose=False)
    # `fit` si `predict` se cronometreaza SEPARAT. `fit` e preprocesare pe CPU (power, quantile) si
    # nu se inmulteste cu n_estimators; `predict` e partea de GPU si se inmulteste. Amestecate, cum
    # erau, partea de CPU intra de opt ori in estimare in loc de o data.
    t0 = time.time(); reg.fit(A[sel], b[sel]); t_fit = time.time() - t0
    t0 = time.time()
    pred = np.asarray(reg.predict(P[:npred]), dtype=np.float64)
    t_pred = time.time() - t0
    act = torch.cuda.max_memory_allocated() / 2**30 - BAZA   # DOAR activarile
    fin = int(np.isfinite(pred).sum())
    # Latimea ADEVARATA a lotului, citita din generatorul potrivit, nu din variabila de mediu.
    HC_REAL = max(len(sp) for cfgs in reg.ensemble_generator_.ensemble_configs_.values()
                  for sp, _, _, _ in cfgs)
    N_NORM_CAL = len(reg.ensemble_generator_.preprocessors_)
    masurat.append((n_ctx + npred, t_pred, t_fit, act))
    print(f'  T={n_ctx + npred:5d}  predict {t_pred:6.1f}s  fit {t_fit:5.1f}s  '
          f'activari {act:5.2f} GiB  finite {fin}/{len(pred)}')
    if fin != len(pred):
        raise SystemExit('predictii neifinite deja la proba: opreste-te, e dtype-ul.')

(T1, t1, tf1, a1), (T2, t2, tf2, a2) = masurat
print(f'\nlatime pe care o vede modelul: {HC_REAL} (MAXFEAT={HC}'
      + (f' + {HC_REAL - HC} de la crosses)' if HC_REAL > HC else ')'))
if HC_REAL == HC and ans['n_feature_crosses'] not in (0, '0', None):
    print('  ATENTIE: crosses e pornit dar latimea n-a crescut. Daca aici scrie acelasi numar ca'
          '\n  MAXFEAT, calibrarea inca ruleaza cu un singur membru si cifra de memorie de mai jos'
          '\n  e cu ~6% optimista.')
# TIMPUL: podea la 1.0. Preprocesarea per cerere nu depinde de randuri, deci la probe mici trag
# exponentul potrivit sub cel adevarat si estimarea ar iesi optimista.
et = max(np.log(t2 / t1) / np.log(T2 / T1), 1.0)
# MEMORIA: aceeasi podea ar fi o greseala, si una scumpa. Varful nu e o lege de putere in
# randuri, e o suma de doi termeni: unul FIX, dictat de TABFM_CHUNK, fiindca bufferele
# chunk-uite nu stiu de T, si unul LINIAR, tensorii de etapa [1, T, HC, 256] care raman vii.
# La T mic partea fixa domina, exponentul in log-log iese langa 0, iar podeaua la 1.0 il
# transforma intr-o supraestimare de cateva ori si intr-un `NU INCAPE` fals. Se potriveste
# o dreapta, si panta ei se verifica singura: impartita la marimea unui tensor de etapa
# trebuie sa dea ~2, adica exact ce promit patch-urile din celula 6.
kk = max((a2 - a1) / (T2 - T1), 0.0)                                    # GiB pe rand
c = max(a1 - kk * T1, 0.0)                                              # partea fixa, de chunk
etalon = HC_REAL * 256 * torch.empty(0, dtype=dt).element_size() / 2**30  # un tensor de etapa/rand
n_est = int(os.environ['TABFM_ESTIMATORS'])
Tmax = max(T.values())
f_nnls = 2.0 if ans['enable_nnls'] else 1.0
if ans['enable_nnls']:
    print('nnls pornit: estimarile includ pasajul in plus (~2x)')

print(f'\nexponent in randuri pentru timp: {et:.2f}   (1.0 = liniar)')
print(f'memorie: {c:.2f} GiB ficsi (chunk) + {kk * 2**30 / 1e6:.2f} MB/rand = '
      f'{kk / etalon:.2f} tensori de etapa vii' + ('  <- patch-urile promit 2' if kk > 0 else ''))
if T2 >= Tmax:
    print(f'punctul al doilea e la T={T2}, adica cererea intreaga: memoria de mai jos e MASURATA')
else:
    print(f'estimarile extrapoleaza de {Tmax / T2:.1f}x peste punctul masurat: ordin de marime')

# --- memoria ---------------------------------------------------------------------------
act_max = c + kk * Tmax
varf = BAZA + act_max
print(f'\nvarf estimat la T={Tmax}: {BAZA:.2f} greutati + {act_max:.2f} activari '
      f'= {varf:.2f} GiB din {TOTAL_GIB:.1f}')
plafon = TOTAL_GIB * 0.92
if varf > plafon:
    print(f'  NU INCAPE. Scara de coborare, si primele doua NU ating cifrele:')
    print(f'    1. TABFM_ATTN_GIB {os.environ.get("TABFM_ATTN_GIB", "0.5")} -> jumatate.'
          '\n       Taierea pe intervale de query e exacta algebric (softmax e pe randul de query).')
    print(f'    2. TABFM_CHUNK {os.environ.get("TABFM_CHUNK", "implicit")} -> jumatate.'
          '\n       Scade partea fixa de mai sus; `torch.cat` reasambleaza acelasi tensor.')
    # Scade doar partea liniara: cum se micsoreaza partea fixa cu latimea nu s-a masurat,
    # deci o tinem constanta si iese o latime conservatoare.
    real_nou = int(max(0.0, plafon - BAZA - c) / (kk * Tmax) * HC_REAL) if kk > 0 else 0
    nou = int(real_nou - np.sqrt(max(real_nou, 1))) // 16 * 16      # inapoi din latime reala in MAXFEAT
    if nou >= 64:
        print(f'    3. abia apoi TABFM_MAXFEAT de la {HC} la ~{nou} (latime reala ~{real_nou}).')
    else:
        print(f'    3. nicio latime rezonabila nu ajuta ({real_nou} coloane reale ar fi pragul).')
        if str(dt) != 'torch.bfloat16':
            print('       Esti pe float32: greutatile singure ocupa 6.14 GiB fata de 3.07 in bf16.'
                  '\n       Verifica de ce n-a trecut bf16 proba, ala e primul lucru de reparat.')
        else:
            print('       Placa asta nu duce configuratia. Runtime L4 sau A100 (8.9 / 8.0).')
else:
    print(f'  incape, cu {plafon - varf:.2f} GiB marja pana la 92% din placa')

# --- timpul ----------------------------------------------------------------------------
# Per membru e cifra care se compara cu reperul: rularea de la MAXFEAT=160 a facut 142 min pentru
# 12 cereri x 8 membri, adica ~92s pe membru la T=14107. Cu exponentul 2.70 pe coloane si latimile
# REALE (172 -> 272), factorul e ~3.4, deci asteptarea la 256 e ~5.3 min pe membru.
per_membru = (t2 / N_CAL) * (Tmax / T2) ** et
# `fit` se face o data per cerere, dar in rularea reala peste TOATE familiile de normalizare, nu
# doar peste cele N_CAL folosite aici.
n_norm_real = min(n_est, len(ans['norm_methods'] or ['none', 'power']))
per_fit = tf2 * n_norm_real / max(N_NORM_CAL, 1)
per_cerere = (per_membru * n_est + per_fit) * f_nnls
print(f'\nper membru: ~{per_membru / 60:.1f} min   (reper la MAXFEAT=160: ~1.5 min)')
print(f'per cerere: ~{per_cerere / 60:.0f} min  = {n_est} membri x {per_membru / 60:.1f} min '
      f'+ {per_fit / 60:.1f} min preprocesare pe CPU')
if per_cerere > 30 * 60:
    print('  Peste 30 min pe cerere. Cu checkpointul pe membru din `tabfm_bridge` o deconectare'
          f'\n  pierde doar membrul in curs (~{per_membru / 60:.1f} min), nu cererea intreaga.')
total = sum((t2 / N_CAL) * (t / T2) ** et * n_est + per_fit for t in T.values()) * f_nnls
print(f'toate 12, la n_estimators={n_est}: ~{total / 3600:.1f} ore')
for kx in (4, 6, 16):
    if kx != n_est:
        alt = sum((t2 / N_CAL) * (t / T2) ** et * kx + per_fit for t in T.values()) * f_nnls
        print(f'  la n_estimators={kx}: ~{alt / 3600:.1f} ore')
print(f'\nsesiuni Colab gratuit (~3h utile fiecare): ~{np.ceil(total / 3600 / 2.8):.0f}. '
      f'Celula 8 se re-ruleaza si sare peste ce e gata.')

del model, reg
torch.cuda.empty_cache()


[tabfm] patch-uri: atentie 9.5e-07 (prag 1e-5, buget 0.5 GiB), memorie exact 0, checkpoint pe membru instalat
latime X: 400 coloane, din care un membru vede 256
ansamblu: {'norm_methods': ['none', 'power', 'quantile_rtdl', 'robust'], 'max_num_features': 256, 'n_feature_crosses': 'sqrt', 'n_svd_features': 0, 'enable_nnls': False, 'outlier_threshold': 4.0}
cele 12 cereri, lungimea secventei (context + de prezis):
  ca              14107
  en              14107
  es              14107
  fil             14107
  fr              14107
  it              14107
  ja              14107
  pt              14107
  si              14107
  de              14093
  en_only_test    14047
  en_only_trial    8716


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights from local directory

incarcare model: 280s  |  greutati pe placa 3.08 GiB din 14.6
  (timpul asta e reperul pentru HF_HOME in Drive vs descarcare directa)
  T= 3200  predict   49.9s  fit   3.4s  activari  4.98 GiB  finite 200/200
  T=14107  predict  265.7s  fit  10.0s  activari  7.91 GiB  finite 505/505

latime pe care o vede modelul: 272 (MAXFEAT=256 + 16 de la crosses)

exponent in randuri pentru timp: 1.13   (1.0 = liniar)
memorie: 4.12 GiB ficsi (chunk) + 0.29 MB/rand = 2.07 tensori de etapa vii  <- patch-urile promit 2
punctul al doilea e la T=14107, adica cererea intreaga: memoria de mai jos e MASURATA

varf estimat la T=14107: 3.08 greutati + 7.91 activari = 10.99 GiB din 14.6
  incape, cu 2.41 GiB marja pana la 92% din placa

per membru: ~2.2 min   (reper la MAXFEAT=160: ~1.5 min)
per cerere: ~18 min  = 8 membri x 2.2 min + 0.3 min preprocesare pe CPU
toate 12, la n_estimators=8: ~3.5 ore
  la n_estimators=4: ~1.8 ore
  la n_estimators=6: ~2.6 ore
  la n_estima

In [10]:
import gc, sys, torch
for _a in ('last_traceback', 'last_value', 'last_type', 'last_exc'):
    if hasattr(sys, _a):
        setattr(sys, _a, None)
gc.collect(); torch.cuda.empty_cache()
print(f'{torch.cuda.memory_allocated()/2**30:.2f} GiB alocati, '
      f'{torch.cuda.memory_reserved()/2**30:.2f} rezervati')


3.08 GiB alocati, 3.11 rezervati


## 8. Rularea

Cele 12 cereri, cu un singur model incarcat. `tabfm_bridge` face intai o proba de sanatate pe
date sintetice, deci daca dtype-ul e gresit afli in cateva secunde, nu dupa prima cerere
adevarata. Daca sesiunea cade, re-ruleaza celula: ce e deja in Drive se sare. Protectia e pe doua niveluri:
un `.npy` per cerere in `DEST/`, si un `.npy` per membru de ansamblu in `DEST/_ckpt/<cheie>/`,
deci o taiere la mijlocul unei cereri pierde un membru, nu 42 de minute. La reluare `fit` se reface
intreg (cateva minute de CPU, membrii trebuie regenerati ca sa se stie carui membru ii apartine
fiecare fisier), dar partea de GPU se sare. `TABFM_CKPT=0` stinge nivelul al doilea.

Fisierele de membru raman pe disc si dupa ce cererea se incheie, cateva zeci de KB fiecare: din ele
iese gratis ablatia pe `n_estimators`, adica media primilor k membri, fara nicio rulare noua.

Colab taie sesiunile gratuite; daca estimarea de mai sus trece de cateva ore, ori scazi
`n_estimators`, ori te bazezi pe reluare. Ordinea de rulare pune limbile primele si bratele
EN-only ultimele exact pentru asta: o sesiune care cade dupa cateva ore lasa pe disc partea de
care are nevoie poarta de frontiera.

In [10]:
import numpy as np
from tabfm_bridge import ruleaza

# `device` NU are implicit cuda: tabfm_bridge.ruleaza citeste TABFM_DEVICE si cade pe
# "cpu" daca lipseste. Se da explicit aici, nu prin mediu, ca sa nu depinda de faptul ca
# celula 5 a fost re-rulata dupa un restart de runtime.
ruleaza(np.load(BLOCURI), DEST, device='cuda')

[tabfm] patch-uri: atentie 9.5e-07 (prag 1e-5, buget 0.5 GiB), memorie exact 0, checkpoint pe membru instalat


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights from local directory
[tabfm]   membru 1/3 in 1s (cerere 0m)
[tabfm]   membru 2/3 in 1s (cerere 0m)
[tabfm]   membru 3/3 in 1s (cerere 0m)
[tabfm]   membru 1/3 de pe disc
[tabfm]   membru 2/3 de pe disc
[tabfm]   membru 3/3 in 1s (cerere 0m)
[tabfm]   membru 1/3 de pe disc
[tabfm]   membru 2/3 de pe disc
[tabfm]   membru 3/3 de pe disc
[tabfm] checkpoint pe membru: 3 membri scrisi, reluare partiala 1 recalculat, reluare completa 0, toate patru bit-identice
[tabfm] chunk de randuri 1024 (implicit 4096, ales pentru 40 GB)
[tabfm] model pe cuda/torch.bfloat16 in 295s | 12 cereri, n_estimators=8, max_rows=None, chunk=1024
[tabfm] ansamblu: norm=['none', 'power', 'quantile_rtdl', 'robust'], max_num_features=256, crosses=sqrt, svd=0, nnls=False, outlier=4.0
[tabfm] checkpoint pe membru: /content/drive/MyDrive/tabfm/preds_proj394_maxfeat256/_ckpt
[tabfm]  1/12 ca             deja pe disc, sar
[tabfm]  2/12 de             deja pe disc, sar
[tabfm]  3/12 en             deja pe di

## Status

In [8]:
import os, glob
gata = glob.glob(os.path.join(DEST, '*.npy'))
part = glob.glob(os.path.join(DEST, '_ckpt', '*', '*.npy'))
print(f'{len(gata)}/12 cereri terminate, {len(part)} membri pe disc')
for d in sorted(glob.glob(os.path.join(DEST, '_ckpt', '*'))):
    n = len(glob.glob(os.path.join(d, '*.npy')))
    if not os.path.exists(os.path.join(DEST, os.path.basename(d) + '.npy')):
        print(f'  cerere in curs: {n}/8 membri')


8/12 cereri terminate, 68 membri pe disc
  cerere in curs: 4/8 membri


In [9]:
import os, glob, re, time
import numpy as np

ck = os.path.join(DEST, '_ckpt')
gata = {os.path.basename(p)[:-4] for p in glob.glob(os.path.join(DEST, '*.npy'))}
dirs = sorted(glob.glob(os.path.join(ck, '*')))
N_EST = int(os.environ.get('TABFM_ESTIMATORS', '8'))
probleme = []

print(f'{len(gata)}/12 cereri terminate, {len(dirs)} directoare de checkpoint\n')
for d in dirs:
    k = os.path.basename(d)
    f = sorted(glob.glob(os.path.join(d, '*.npy')))
    tmp = glob.glob(os.path.join(d, '*.tmp.npy'))
    idx = sorted(int(m.group(1)) for m in
                 (re.search(r'_m(\d+)_', os.path.basename(x)) for x in f) if m)
    ultim = max((os.path.getmtime(x) for x in f), default=0)
    stare = 'TERMINATA' if k in gata else 'in curs   '
    print(f'{stare} {k[:12]}  {len(f)} fisiere, membri {sorted(set(idx))}, '
          f'ultimul acum {(time.time()-ultim)/60:.0f} min')
    if len(f) > N_EST:
        probleme.append(f'{k[:12]}: {len(f)} fisiere pentru {N_EST} membri -> '
                        f'AMPRENTE DIFERITE, membrii NU se regenereaza la fel')
    if len(set(idx)) != len(idx):
        probleme.append(f'{k[:12]}: acelasi indice de membru apare de mai multe ori')
    if k in gata and len(set(idx)) != N_EST:
        probleme.append(f'{k[:12]}: terminata dar are {len(set(idx))} membri, nu {N_EST}')
    if tmp:
        probleme.append(f'{k[:12]}: {len(tmp)} fisiere .tmp ramase (scriere intrerupta)')
    if any(os.path.getsize(x) == 0 for x in f):
        probleme.append(f'{k[:12]}: fisiere de 0 octeti')

# Controlul care leaga cele doua niveluri: .npy-ul final TREBUIE sa fie o transformare
# afina a mediei membrilor (mean(membri) apoi inverse_transform al scalarului de y),
# deci corelatia dintre ele e exact 1.0. Daca nu e, membrii de pe disc nu sunt ai cererii.
for d in dirs:
    k = os.path.basename(d)
    p = os.path.join(DEST, k + '.npy')
    if not os.path.exists(p):
        continue
    m = [np.load(x) for x in sorted(glob.glob(os.path.join(d, '*.npy')))]
    med = np.mean([x.reshape(-1) for x in m], axis=0)
    fin = np.load(p)
    if len(med) != len(fin):
        probleme.append(f'{k[:12]}: {len(med)} valori in membri, {len(fin)} in final')
        continue
    r = np.corrcoef(med, fin)[0, 1]
    if r < 0.999999:
        probleme.append(f'{k[:12]}: corelatia media-membri vs final e {r:.6f}, nu 1.0')

print('\n' + ('TOTUL E CONSISTENT' if not probleme else 'PROBLEME:'))
for x in probleme:
    print('  -', x)


8/12 cereri terminate, 9 directoare de checkpoint

TERMINATA 14a45a598d6a  8 fisiere, membri [0, 1, 2, 3, 4, 5, 6, 7], ultimul acum 1044 min
TERMINATA 35e773ad5616  8 fisiere, membri [0, 1, 2, 3, 4, 5, 6, 7], ultimul acum 1008 min
TERMINATA 3f5019857550  8 fisiere, membri [0, 1, 2, 3, 4, 5, 6, 7], ultimul acum 972 min
in curs    472dc51bc9b2  4 fisiere, membri [0, 1, 2, 3], ultimul acum 909 min
TERMINATA 58ade66c95a4  8 fisiere, membri [0, 1, 2, 3, 4, 5, 6, 7], ultimul acum 1026 min
TERMINATA 61002293644e  8 fisiere, membri [0, 1, 2, 3, 4, 5, 6, 7], ultimul acum 936 min
TERMINATA c3bf5c0da3a8  8 fisiere, membri [0, 1, 2, 3, 4, 5, 6, 7], ultimul acum 990 min
TERMINATA ead6312e51d6  8 fisiere, membri [0, 1, 2, 3, 4, 5, 6, 7], ultimul acum 954 min
TERMINATA ec02f7acaa95  8 fisiere, membri [0, 1, 2, 3, 4, 5, 6, 7], ultimul acum 918 min

TOTUL E CONSISTENT


## 9. Inapoi pe local

Descarca folderul `preds_proj394` din Drive in `data/tabfm_preds_proj394/` si ruleaza:

```
TABFM_CACHE=data/tabfm_preds_proj394 python finetune_xlmr.py --frozentrial \
    --lemma-freq --afeat fert_inloc --emb-tf proj-394 --regressor tabfm-cache
```

Daca o cheie lipseste sau blocurile difera de cele exportate, comanda se opreste si nu scrie
nimic. Ia si `setari.json`: acolo scrie cu ce `n_estimators` si ce dtype s-a rulat, si fara el
cifra nu se poate atribui unei configuratii.

Celula de mai jos face o arhiva, daca preferi sa o iei ca un singur fisier.

In [11]:
import shutil
# Numele se deriva din DEST. Fixat pe 'preds_proj394', arhiva de la 256 iesea cu numele celei
# de la 160 si cele doua se amestecau in folderul de descarcari.
z = shutil.make_archive(os.path.join('/content', os.path.basename(DEST)), 'zip', DEST)
print(z, os.path.getsize(z) / 1e6, 'MB')
from google.colab import files
files.download(z)

/content/preds_proj394_maxfeat256.zip 0.279353 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>